
### Layer 3 — The Impact
*Tables: loans + education*

> Did World Bank education loans actually move the needle on
> enrollment and literacy — or did the money disappear without a trace?

**Questions:**
1. Did **primary and secondary enrollment** improve in countries that received education loans?
2. Which countries improved the **most vs least** on literacy after receiving loans?
3. Is there a **correlation** between total loan amount received and education improvement?
4. Did improvement happen **during the loan period** or only after?
5. Which **income group** saw the most education gains from loans?


In [1]:
import pandas as pd

loans = pd.read_csv("../data/cleaned/loans_clean.csv")
education = pd.read_csv("../data/cleaned/education_clean.csv")

print(f"loans: {loans.shape}")
print(f"education: {education.shape}")

loans: (11236, 30)
education: (1500, 6)


In [16]:
countries = pd.read_csv("../data/cleaned/countries_clean.csv")

# Use countries as a bridge to get ISO2 and ISO3 in the same place
loans_with_iso3 = loans.merge(
    countries[["country_code_iso2", "country_code_iso3", "income_level"]],
    left_on="country_code",
    right_on="country_code_iso2",
    how="left"
)

loans_edu = loans_with_iso3.merge(
    education,
    left_on="country_code_iso3",
    right_on="countryiso3code",
    how="inner"
)

print(f"Shape after merge: {loans_edu.shape}")
print(f"Unique countries: {loans_edu['country_y'].nunique()}")
print(f"Columns: {loans_edu.columns.tolist()}")

Shape after merge: (76050, 39)
Unique countries: 26
Columns: ['agreement_signing_date', 'board_approval_date', 'borrower', 'borrowers_obligation_usd', 'cancelled_amount_usd', 'closed_date_most_recent', 'country_x', 'country_code', 'credit_number', 'credit_status', 'credits_held_usd', 'currency_of_commitment', 'disbursed_amount_usd', 'due_3rd_party_usd', 'due_to_ida_usd', 'effective_date_most_recent', 'end_of_period', 'exchange_adjustment_usd', 'first_repayment_date', 'last_disbursement_date', 'last_repayment_date', 'original_principal_amount_usd', 'project_id', 'project_name', 'region', 'repaid_3rd_party_usd', 'repaid_to_ida_usd', 'service_charge_rate', 'sold_3rd_party_usd', 'undisbursed_amount_usd', 'country_code_iso2', 'country_code_iso3', 'income_level', 'indicator', 'country_y', 'countryiso3code', 'date', 'value', 'indicator_code']


In [5]:
print(f"Shape: {loans_edu.shape}")
print(f"Unique countries: {loans_edu['country_x'].nunique()}")
print(f"Unique indicators: {loans_edu['indicator'].unique()}")
print(f"Date range: {loans_edu['date'].min()} to {loans_edu['date'].max()}")

Shape: (76050, 39)
Unique countries: 26
Unique indicators: ['School enrollment, primary (% gross)'
 'School enrollment, secondary (% gross)'
 'Literacy rate, adult total (% of people ages 15 and above)']
Date range: 2015 to 2025


In [6]:
print(loans_edu[["country_x", "credit_number", "original_principal_amount_usd", 
                  "indicator", "date", "value"]].head(10).to_string())

  country_x credit_number  original_principal_amount_usd                             indicator  date       value
0     Chile      IDA00040                    22878919.07  School enrollment, primary (% gross)  2025         NaN
1     Chile      IDA00040                    22878919.07  School enrollment, primary (% gross)  2024   99.494232
2     Chile      IDA00040                    22878919.07  School enrollment, primary (% gross)  2023  100.318382
3     Chile      IDA00040                    22878919.07  School enrollment, primary (% gross)  2022  100.192268
4     Chile      IDA00040                    22878919.07  School enrollment, primary (% gross)  2021   99.365227
5     Chile      IDA00040                    22878919.07  School enrollment, primary (% gross)  2020   99.704399
6     Chile      IDA00040                    22878919.07  School enrollment, primary (% gross)  2019  101.193207
7     Chile      IDA00040                    22878919.07  School enrollment, primary (% gross)  

In [9]:
# Aggregate total loan investment per country
loans_by_country = loans.merge(
    countries[["country_code_iso2", "country_code_iso3", "income_level"]],
    left_on="country_code",
    right_on="country_code_iso2",
    how="left"
).groupby(["country_code_iso3", "income_level"]).agg(
    total_loans=("original_principal_amount_usd", "sum"),
    loan_count=("credit_number", "count"),
    total_disbursed=("disbursed_amount_usd", "sum")
).reset_index()

# Now join with education
loans_edu = loans_by_country.merge(
    education,
    left_on="country_code_iso3",
    right_on="countryiso3code",
    how="inner"
)

print(f"Shape: {loans_edu.shape}")
print(f"Unique countries: {loans_edu['country'].nunique()}")
print(loans_edu.head(3).to_string())

Shape: (780, 11)
Unique countries: 26
  country_code_iso3 income_level   total_loans  loan_count  total_disbursed                             indicator      country countryiso3code  date       value indicator_code
0               AFG   Low income  6.385147e+09         136     4.706529e+09  School enrollment, primary (% gross)  Afghanistan             AFG  2025         NaN    SE.PRM.ENRR
1               AFG   Low income  6.385147e+09         136     4.706529e+09  School enrollment, primary (% gross)  Afghanistan             AFG  2024         NaN    SE.PRM.ENRR
2               AFG   Low income  6.385147e+09         136     4.706529e+09  School enrollment, primary (% gross)  Afghanistan             AFG  2023  112.230904    SE.PRM.ENRR


In [8]:
print(loans_edu.columns.tolist())

['country_code_iso3', 'income_level', 'total_loans', 'loan_count', 'total_disbursed', 'indicator', 'country', 'countryiso3code', 'date', 'value', 'indicator_code']


The problem with the first merge:
The first merge joined every single loan row directly with every education row for the same country. So if Chile has 5 loans and 10 years of education data, i'd get 5 × 10 = 50 rows for Chile , with the same loan amount repeated 50 times. That's wrong because:

>You can't say a 1961 loan caused a 2024 enrollment rate

Did primary and secondary enrollment improve in countries that received education loans?

In [13]:
primary = loans_edu[loans_edu["indicator"] == "School enrollment, primary (% gross)"]

primary_trend = primary.groupby("date").agg(
    avg_enrollment=("value", "mean")
).reset_index()

print(primary_trend.to_string())

   date  avg_enrollment
0  2016      102.386245
1  2017      101.684872
2  2018      101.017823
3  2019      100.622076
4  2020      101.228228
5  2021       99.283061
6  2022      100.608076
7  2023       98.289009
8  2024      100.867679
9  2025      100.627548


In [14]:
secondary = loans_edu[loans_edu["indicator"] == "School enrollment, secondary (% gross)"]

secondary_trend = secondary.groupby("date").agg(
    avg_enrollment=("value", "mean")
).reset_index()

print(secondary_trend.to_string())

   date  avg_enrollment
0  2016       70.157565
1  2017       71.048276
2  2018       76.553972
3  2019       79.495577
4  2020       80.247882
5  2021       76.833998
6  2022       79.146542
7  2023       73.944697
8  2024       78.683683
9  2025       23.125601


In [15]:
primary_trend = primary_trend[primary_trend["date"] < 2025]
secondary_trend = secondary_trend[secondary_trend["date"] < 2025]

print("Primary enrollment trend:")
print(primary_trend.to_string())
print("\nSecondary enrollment trend:")
print(secondary_trend.to_string())

Primary enrollment trend:
   date  avg_enrollment
0  2016      102.386245
1  2017      101.684872
2  2018      101.017823
3  2019      100.622076
4  2020      101.228228
5  2021       99.283061
6  2022      100.608076
7  2023       98.289009
8  2024      100.867679

Secondary enrollment trend:
   date  avg_enrollment
0  2016       70.157565
1  2017       71.048276
2  2018       76.553972
3  2019       79.495577
4  2020       80.247882
5  2021       76.833998
6  2022       79.146542
7  2023       73.944697
8  2024       78.683683


Among the 26 countries in our dataset, secondary enrollment showed meaningful improvement over 2016-2024. But we can't fully credit World Bank loans, this is a global trend driven by many factors.

Which countries improved the most vs least on literacy after receiving loans?

In [25]:
enroll = loans_edu[
    loans_edu["indicator"].isin([
        "School enrollment, primary (% gross)",
        "School enrollment, secondary (% gross)"
    ]) & (loans_edu["date"] < 2025)
]

early = enroll[enroll["date"] <= 2019].groupby("country_y")["value"].mean()
recent = enroll[enroll["date"] >= 2020].groupby("country_y")["value"].mean()

improvement = pd.DataFrame({
    "early_avg": early,
    "recent_avg": recent
}).dropna()

improvement["change"] = (improvement["recent_avg"] - improvement["early_avg"]).round(2)
improvement = improvement.sort_values("change", ascending=False)

print(improvement.to_string())

                         early_avg  recent_avg  change
country_y                                             
Angola                   54.416799   70.145265   15.73
Chad                     50.960917   60.118525    9.16
Bangladesh               78.202688   86.439629    8.24
Cote d'Ivoire            69.131688   74.192978    5.06
Bhutan                   93.599031   98.143806    4.54
Armenia                  90.086354   93.659549    3.57
Costa Rica              115.086230  118.342260    3.26
Cambodia                 78.780004   81.145649    2.37
Bosnia and Herzegovina   91.630960   93.796371    2.17
China                    97.511144   99.386084    1.87
Comoros                  82.983224   84.174992    1.19
Bolivia                  93.042647   93.970129    0.93
Afghanistan              85.202839   85.922253    0.72
Chile                   102.134309  102.694757    0.56
Congo, Rep.              87.462112   87.880241    0.42
Cabo Verde              100.943151  100.195681   -0.75
Albania   

In [28]:
print(education[education["country"] == "Morocco"])
# World Bank API for Morocco education indicators
# http://api.worldbank.org/v2/country/MAR/indicator/SE.PRM.ENRR?format=json&date=2015:2025

Empty DataFrame
Columns: [indicator, country, countryiso3code, date, value, indicator_code]
Index: []


Is there a correlation between total loan amount received and education improvement?

In [30]:
early = enroll[enroll["date"] <= 2019].groupby(
    ["country_y", "countryiso3code"])["value"].mean()
recent = enroll[enroll["date"] >= 2020].groupby(
    ["country_y", "countryiso3code"])["value"].mean()

early = early.reset_index()
recent = recent.reset_index()

improvement = early.merge(recent, on=["country_y", "countryiso3code"], suffixes=("_early", "_recent"))
improvement["change"] = (improvement["value_recent"] - improvement["value_early"]).round(2)
improvement = improvement.sort_values("change", ascending=False)

print(improvement.head(5).to_string())

        country_y countryiso3code  value_early  value_recent  change
2          Angola             AGO    54.416799     70.145265   15.73
18           Chad             TCD    50.960917     60.118525    9.16
5      Bangladesh             BGD    78.202688     86.439629    8.24
25  Cote d'Ivoire             CIV    69.131688     74.192978    5.06
8          Bhutan             BTN    93.599031     98.143806    4.54


In [33]:
correlation = improvement.merge(
    loans_by_country[["country_code_iso3", "total_loans", "total_disbursed", "loan_count"]],
    left_on="countryiso3code",
    right_on="country_code_iso3",
    how="left"
)

corr_result = correlation[["change", "total_loans", "total_disbursed", "loan_count"]].corr()
print(corr_result.to_string())

print("\nFull picture:")
print(correlation[["country_y", "change", "total_loans", "loan_count"]].sort_values("change", ascending=False).to_string())

                   change  total_loans  total_disbursed  loan_count
change           1.000000     0.102552         0.129434   -0.135688
total_loans      0.102552     1.000000         0.993201    0.876360
total_disbursed  0.129434     0.993201         1.000000    0.872290
loan_count      -0.135688     0.876360         0.872290    1.000000

Full picture:
                   country_y  change   total_loans  loan_count
0                     Angola   15.73  1.023580e+09          24
1                       Chad    9.16  4.941337e+09         129
2                 Bangladesh    8.24  4.701597e+10         391
3              Cote d'Ivoire    5.06  1.257229e+10         144
4                     Bhutan    4.54  1.325098e+09          51
5                    Armenia    3.57  1.407870e+09          66
6                 Costa Rica    3.26  6.428899e+06           1
7                   Cambodia    2.37  3.522760e+09          84
8     Bosnia and Herzegovina    2.17  1.564740e+09          73
9              

>if there's no correlation between loan amount and education improvement, then asking "did improvement happen during or after the loan period" becomes irrelevant. You can't time something that has no effect to measure.
Skip Q4, move to Q5:


Which income group saw the most education gains from loans?

In [35]:
correlation = improvement.merge(
    loans_by_country[["country_code_iso3", "total_loans", "total_disbursed", "loan_count", "income_level"]],
    left_on="countryiso3code",
    right_on="country_code_iso3",
    how="left"
)

income_gains = correlation.groupby("income_level").agg(
    avg_change=("change", "mean"),
    country_count=("change", "count")
).round(2).sort_values("avg_change", ascending=False)

print(income_gains.to_string())

                     avg_change  country_count
income_level                                  
High income                1.91              2
Low income                 0.26              4
Lower middle income       -0.50             10
Upper middle income       -0.95              8


>High income countries improved the most, while middle income countries actually got worse on average.

### Layer 3 — The Impact: Findings Summary

#### Core Question
> Did World Bank education loans actually move the needle on
> enrollment and literacy — or did the money disappear without a trace?

#### Important Limitations Before Reading
- Education data covers only 26 countries where both loan and
  education data exist
- Analysis period is 2015 to 2024 (2025 dropped — incomplete reporting)
- We cannot prove direct causation between loans and education outcomes
  — only correlation
- COVID-19 disrupted education globally in 2020-2021, affecting trends


#### Q1 — Did Enrollment Improve Over Time?

**Primary enrollment trend (2016-2024):**
| Period | Average Enrollment |
|---|---|
| 2016 | 102.39% |
| 2020 | 101.23% |
| 2024 | 100.87% |

**Secondary enrollment trend (2016-2024):**
| Period | Average Enrollment |
|---|---|
| 2016 | 70.16% |
| 2020 | 80.25% |
| 2024 | 78.68% |

**Key insight:**
Primary enrollment is essentially flat — hovering between 98% and 102%
the entire period. These 26 countries have already achieved near
universal primary education. There is no room left to improve here,
which means World Bank loans likely contributed to reaching this
ceiling decades ago, not recently.

Secondary enrollment tells a more interesting story. It improved by
10 percentage points between 2016 and 2020 — a meaningful gain.
Then COVID hit in 2021, dropping it back down. By 2024 it had
partially recovered to 78.7%. The net improvement over the full
period is 8.5 percentage points — real progress, but driven by
global trends not just loans.


#### Q2 — Which Countries Improved the Most vs Least?

**Most improved (2016-2019 vs 2020-2024):**
| Country | Change | Total Loans | Story |
|---|---|---|---|
| Angola | +15.73 | 1 billion USD | Post-conflict recovery + WB investment |
| Chad | +9.16 | 4.9 billion USD | Starting from very low base |
| Bangladesh | +8.24 | 47 billion USD | Consistent development success |
| Cote d'Ivoire | +5.06 | 12.6 billion USD | Recovering from political instability |
| Bhutan | +4.54 | 1.3 billion USD | Small nation, strong governance |

**Most declined:**
| Country | Change | Total Loans | Story |
|---|---|---|---|
| Benin | -27.18 | 7.4 billion USD | Dramatic collapse despite heavy investment |
| Cameroon | -16.26 | 8.1 billion USD | Significant deterioration |
| Belize | -6.32 | 43 million USD | Small island — COVID impact |
| Burkina Faso | -5.93 | 10.3 billion USD | Active Sahel conflict since 2015 |

**Key insight:**
The most alarming finding is Benin and Cameroon — both received
billions in World Bank loans yet saw dramatic education declines.
Meanwhile Angola improved 15 points with far less money. This
already hints that loan amount is not the deciding factor.


##### Q3 — Is There a Correlation Between Loan Amount and Improvement?

**Correlation results:**
| Variable | Correlation with Education Change |
|---|---|
| Total loans | 0.10 |
| Total disbursed | 0.13 |
| Loan count | -0.14 |

**Key insight — the most important finding in Layer 3:**
There is virtually no correlation between how much money a country
borrowed and whether its education outcomes improved.

A correlation of 0.10 is essentially zero. Loan count is slightly
negative — meaning countries with more loans showed marginally worse
outcomes on average.

The evidence speaks clearly:
- Benin: 7.4 billion USD, 204 loans → enrollment dropped 27 points
- Burkina Faso: 10.3 billion USD, 230 loans → dropped 5.9 points
- Angola: 1 billion USD, 24 loans → improved 15.7 points
- Costa Rica: 6.4 million USD, 1 loan → improved 3.3 points

More money does not reliably produce better education outcomes.
The real drivers are political stability, governance quality,
conflict presence, and existing institutional capacity — none of
which a loan can directly fix.

##### Q4 — Skipped

Given the near-zero correlation established in Q3, analyzing
whether improvement happened during or after the loan period
would be meaningless. There is no consistent improvement to time.
This was an analytical decision based on evidence.


##### Q5 — Which Income Group Saw the Most Education Gains?

**Average enrollment change by income group:**
| Income Level | Avg Change | Countries |
|---|---|---|
| High income | +1.91 | 2 |
| Low income | +0.26 | 4 |
| Lower middle income | -0.50 | 10 |
| Upper middle income | -0.95 | 8 |

**Key insight:**
High income countries improved the most — but they only represent
2 countries in our sample, making this statistically weak.

Low income countries showed small positive gains. Starting from
such a low base, even a +0.26 average improvement represents
real children entering classrooms.

Middle income countries went backwards on average. These countries
are in a transition phase — rapid urbanization, demographic
pressures, and economic disruption are affecting education systems
faster than loans can compensate.


#### Layer 3 — Overall Verdict

1. Primary enrollment is already solved — near universal across
   all 26 countries. Loans may have helped decades ago but have
   nothing left to fix here.

2. Secondary enrollment improved meaningfully from 2016 to 2020
   before COVID disrupted progress. Recovery is underway but incomplete.

3. There is no meaningful correlation between loan amount and
   education improvement. Money alone does not fix education.

4. The worst performers — Benin, Cameroon, Burkina Faso — received
   some of the most loans. The best performers — Angola, Costa Rica
   — received relatively little.

5. Conflict and instability are the real enemies of education
   progress, not lack of funding.

> World Bank education loans reach the classroom in some countries
> and disappear into instability in others. The loan is neutral —
> what determines outcomes is the environment it lands in.
> Pouring money into a country at war or with broken institutions
> does not build schools. It builds debt.